# SAM：Segment Anything Model

这个 Notebook 展示 `SAM（Segment Anything Model）` 的交互式图像分割流程。

内容包括：
- SAM 三大组件解读：Image Encoder / Prompt Encoder / Mask Decoder
- 三种 prompt 模式：点击 prompt、bounding box prompt、自动分割
- 与 YOLO / DETR（检测）的任务粒度对比
- Mask 多彩可视化

## 1. 环境准备

```bash
pip install torch transformers pillow requests matplotlib numpy
```

In [ ]:
from dataclasses import dataclass
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from PIL import Image
from transformers import SamModel, SamProcessor

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # SAM-ViT-Base（最轻量，适合本地运行）
    model_name: str = 'facebook/sam-vit-base'
    # 每个 prompt 预测多个候选 mask，取置信度最高的
    multimask_output: bool = True

cfg = Config()
cfg

## 2. 加载示例图像

In [ ]:
def load_image_from_url(url):
    resp = requests.get(url, timeout=10)
    return Image.open(BytesIO(resp.content)).convert('RGB')


image = load_image_from_url('http://images.cocodataset.org/val2017/000000039769.jpg')
image_np = np.array(image)

plt.figure(figsize=(7, 5))
plt.imshow(image)
plt.axis('off')
plt.title('输入图像')
plt.show()
print(f'图像尺寸：{image.size}  数组形状：{image_np.shape}')

## 3. 模型与 Processor 加载

In [ ]:
processor = SamProcessor.from_pretrained(cfg.model_name)
model = SamModel.from_pretrained(cfg.model_name).to(device)
model.eval()
print('SAM 加载完成')

## 4. SAM 结构解读

### 4.1 三大组件

| 组件 | 结构 | 作用 |
|------|------|------|
| Image Encoder | MAE 预训练的 ViT-H/L/B | 将图像编码为密集特征图（只需运行一次） |
| Prompt Encoder | 点/框/文本的嵌入层 | 将用户交互信号编码为 prompt embedding |
| Mask Decoder | 轻量 Transformer（2层） | 结合图像特征与 prompt，预测分割 mask |

### 4.2 高效推理设计

Image Encoder 计算量最大，但**只需对图像运行一次**，编码结果可缓存。
不同 prompt（点、框）的 Mask Decoder 推理极快（毫秒级），使交互式分割成为可能。

### 4.3 与检测模型的对比

| 维度 | YOLO / DETR | SAM |
|------|-------------|-----|
| 任务 | 目标检测（边界框 + 类别） | 实例分割（精确 mask） |
| 类别感知 | 是（closed-set） | 否（class-agnostic，只分割，不分类） |
| 交互性 | 无 | 支持点击、框选等交互 prompt |
| 适用场景 | 快速检测已知类别 | 任意对象的精确分割 |

## 5. 点击 Prompt 分割演示

In [ ]:
def show_mask(mask, ax, color=None, alpha=0.5):
    if color is None:
        color = np.array([30/255, 144/255, 255/255])
    h, w = mask.shape
    mask_img = np.zeros((h, w, 4))
    mask_img[mask] = np.append(color, alpha)
    ax.imshow(mask_img)


def show_point(point, label, ax, marker_size=200):
    color = 'green' if label == 1 else 'red'
    ax.scatter(point[0], point[1], c=color, s=marker_size, marker='*', zorder=5)


@torch.no_grad()
def segment_with_points(model, processor, image, points, labels, device):
    # points 格式：[[x, y], ...],  labels: 1=前景, 0=背景
    inputs = processor(
        images=image,
        input_points=[[points]],
        input_labels=[[labels]],
        return_tensors='pt'
    ).to(device)

    outputs = model(**inputs, multimask_output=cfg.multimask_output)
    # 取置信度最高的 mask
    scores = outputs.iou_scores[0]
    best_idx = scores.argmax().item()
    mask = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs['original_sizes'].cpu(),
        inputs['reshaped_input_sizes'].cpu()
    )[0][0][best_idx].numpy()
    return mask, scores[best_idx].item()


# 点击猫的身体（图像坐标 x, y）
points = [[300, 200]]
labels = [1]

mask, score = segment_with_points(model, processor, image, points, labels, device)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(image)
show_point(points[0], labels[0], axes[0])
axes[0].set_title('点击 Prompt（绿星）')
axes[0].axis('off')

axes[1].imshow(image)
show_mask(mask, axes[1])
axes[1].set_title(f'分割结果  IoU score={score:.3f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 6. Bounding Box Prompt 分割演示

In [ ]:
@torch.no_grad()
def segment_with_box(model, processor, image, box, device):
    # box 格式：[x_min, y_min, x_max, y_max]
    inputs = processor(
        images=image,
        input_boxes=[[[box]]],
        return_tensors='pt'
    ).to(device)

    outputs = model(**inputs, multimask_output=False)
    mask = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs['original_sizes'].cpu(),
        inputs['reshaped_input_sizes'].cpu()
    )[0][0][0].numpy()
    score = outputs.iou_scores[0, 0].item()
    return mask, score


# 框选左侧猫咪的大致区域
box = [70, 60, 400, 370]
mask_box, score_box = segment_with_box(model, processor, image, box, device)

import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(image)
rect = mpatches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                           linewidth=2, edgecolor='green', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title('Box Prompt')
axes[0].axis('off')

axes[1].imshow(image)
show_mask(mask_box, axes[1], color=np.array([255/255, 100/255, 30/255]))
axes[1].set_title(f'分割结果  IoU score={score_box:.3f}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 7. 自动分割一切（Everything Mode）

SAM 也支持不给任何 prompt，通过密集采样点自动检测并分割图像中的所有对象。

In [ ]:
@torch.no_grad()
def auto_segment(model, processor, image, device, grid_size=16):
    # 在图像上均匀撒点，让每个点都作为前景 prompt
    W, H = image.size
    xs = np.linspace(0, W, grid_size + 2)[1:-1]
    ys = np.linspace(0, H, grid_size + 2)[1:-1]
    grid_points = [[int(x), int(y)] for y in ys for x in xs]
    grid_labels = [1] * len(grid_points)

    inputs = processor(
        images=image,
        input_points=[[grid_points]],
        input_labels=[[grid_labels]],
        return_tensors='pt'
    ).to(device)

    outputs = model(**inputs, multimask_output=False)
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs['original_sizes'].cpu(),
        inputs['reshaped_input_sizes'].cpu()
    )[0][:, 0].numpy()   # N, H, W

    scores = outputs.iou_scores[0, :, 0].cpu().numpy()
    return masks, scores


masks_auto, scores_auto = auto_segment(model, processor, image, device, grid_size=8)

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(image)
colors = plt.cm.tab20(np.linspace(0, 1, len(masks_auto)))
for mask, color in zip(masks_auto, colors):
    show_mask(mask, ax, color=color[:3], alpha=0.4)
ax.set_title(f'自动分割一切（共 {len(masks_auto)} 个区域）')
ax.axis('off')
plt.tight_layout()
plt.show()

## 8. 与检测模型对比总结

SAM 与同目录中 YOLO / DETR 的定位：

- **YOLO**：极速检测，closed-set 类别，输出边界框
- **DETR**：端到端检测，Transformer 架构，输出边界框
- **SAM**：class-agnostic 分割，任意对象精确 mask，支持人机交互

实际应用中三者互补：先用 YOLO/DETR 定位目标（得到 box），再用 SAM 精化分割（得到 mask）。